# BES-100M-A20M: the 24-hour run on one TPU v6e

This notebook runs `scripts/run_24h.py` with `configs/final.py` on a Colab **TPU v6e-1** runtime:
downloads, tokenisation, the library, the start-up benchmark and canary, about 19 hours of
training, post-training, the proxy evaluation and the export, inside one 24-hour budget of wall
time. Read `RUNBOOK.md` first (accounts, what the first 40 minutes must show, the fallback
ladder, what each failure means). Nothing in this repository has been measured on a TPU
(`docs/TPU_NOTES.md`).

Order: run cells 1, 1b and 2 to 6 once, top to bottom. Cell 7 shows progress (run it as often as you
like). Cell 8 continues the run after a disconnect. Cells 10 and 11 (chat, blind arena) are
for after the run.

The kernel of this notebook never imports JAX before the run: whichever process imports it first
on the TPU host owns the TPU.

In [ ]:
# 1. parameters: edit, then run the cells in order ------------------------------------------------
REPO_URL = ""                      # git URL of this repository (a private one needs a token URL: see RUNBOOK.md)
BRANCH = ""                        # the branch that holds this code ("" = the default branch)
REPO_DIR = "/content/TuringAward2027"   # an existing copy (for example on Drive) is used as it is
RUN_DIR = "/content/runs/final"    # on the local disk; Drive only mirrors it
USE_DRIVE = True                   # mirror checkpoints, ledgers, reports and the export to Drive
DRIVE_DIR = "/content/drive/MyDrive/bes_runs/final"
CONFIG = "configs/final.py"
BUDGET_MINUTES = None              # None: the configuration's 24 hours
ALLOW_CPU = False                  # True only for a rehearsal on a CPU runtime

In [ ]:
# 1b. helpers: run this cell, nothing to edit (no JAX in this kernel: it would take the TPU) -----
import json, os, shutil, signal, subprocess, sys, time

PID_FILE = RUN_DIR.rstrip("/") + ".pid"
LOG = RUN_DIR.rstrip("/") + ".log"
MIRROR_PID = RUN_DIR.rstrip("/") + ".mirror.pid"


def pid_alive(path=PID_FILE):
    """True when the process whose pid is in ``path`` is running."""
    try:
        pid = int(open(path).read().strip())
    except (OSError, ValueError):
        return False
    try:
        os.kill(pid, 0)
    except OSError:
        return False
    try:                                   # a zombie (finished, not reaped) is not running
        return open(f"/proc/{pid}/stat").read().split()[2] != "Z"
    except OSError:
        return True


def sh(cmd, env=None, cwd=None):
    """Run a command, print the tail of its output in this cell, raise with it on failure
    (a child's own output does not reach a notebook cell)."""
    p = subprocess.run(list(map(str, cmd)), capture_output=True, text=True, env=env, cwd=cwd)
    out = (p.stdout + p.stderr).strip()
    if p.returncode != 0:
        raise RuntimeError(f"{' '.join(map(str, cmd))} failed with exit status {p.returncode}:\n{out[-4000:]}")
    if out:
        print(out[-1500:])
    return p


def start_background(argv, log=LOG, pid_file=PID_FILE, env=None):
    """Start ``argv`` in its own session (it survives a kernel restart), output appended to ``log``."""
    os.makedirs(os.path.dirname(log) or ".", exist_ok=True)
    with open(log, "ab") as f:
        f.write(f"\n===== {time.strftime('%Y-%m-%d %H:%M:%S')} {' '.join(argv)}\n".encode())
        proc = subprocess.Popen(argv, stdout=f, stderr=subprocess.STDOUT, cwd=REPO_DIR,
                                env=dict(os.environ, PYTHONUNBUFFERED="1", **(env or {})),
                                start_new_session=True)
    with open(pid_file, "w") as f:
        f.write(str(proc.pid))
    return proc.pid


# Mirror of the run directory to Google Drive: every file but the compilation cache, token
# shards and temporary files; checkpoints pruned locally are pruned on Drive as well.
MIRROR_CODE = r"""
import os, shutil, sys, time
SKIP = {"jax_cache", "data", "tok", "raw", ".supervisor.lock", "__pycache__"}
def skip(name):
    return name in SKIP or name.startswith((".tmp-", ".trash-", ".old-")) or name.endswith(".part")
def sync(src, dst):
    if not os.path.isdir(src):
        return
    for root, dirs, files in os.walk(src):
        rel = os.path.relpath(root, src)
        dirs[:] = [d for d in dirs if not skip(d)]
        os.makedirs(os.path.join(dst, rel), exist_ok=True)
        for f in files:
            if skip(f):
                continue
            s, d = os.path.join(root, f), os.path.join(dst, rel, f)
            try:
                st = os.stat(s)
                if os.path.exists(d) and os.stat(d).st_size == st.st_size and os.stat(d).st_mtime >= st.st_mtime:
                    continue
                shutil.copy2(s, d + ".part")
                os.replace(d + ".part", d)
            except OSError as e:
                print("mirror:", e, flush=True)
        if os.path.basename(root) == "ckpt":          # checkpoints pruned locally go on Drive too
            here = set(os.listdir(root))
            there = os.path.join(dst, rel)
            for name in os.listdir(there):
                if name.startswith("ckpt-") and name not in here:
                    shutil.rmtree(os.path.join(there, name), ignore_errors=True)
if __name__ == "__main__":
    period = float(sys.argv[1])
    pairs = [p.split("=>") for p in sys.argv[2:]]
    while True:
        for src, dst in pairs:
            sync(src, dst)
        print(time.strftime("%H:%M:%S"), "mirrored", flush=True)
        time.sleep(period)
"""


def start_mirror(period_s=600):
    """Mirror the run, the library and the proxy outputs to DRIVE_DIR (when USE_DRIVE)."""
    if not USE_DRIVE:
        print("no Drive mirror (USE_DRIVE = False)")
        return
    if pid_alive(MIRROR_PID):
        print("the Drive mirror is already running")
        return
    code = RUN_DIR.rstrip("/") + ".mirror.py"
    with open(code, "w") as f:
        f.write(MIRROR_CODE)
    pairs = [f"{RUN_DIR}=>{DRIVE_DIR}",
             f"{REPO_DIR}/data_cache/library=>{DRIVE_DIR}.data_cache/library",
             f"{REPO_DIR}/data_cache/proxy=>{DRIVE_DIR}.data_cache/proxy"]
    pid = start_background([sys.executable, code, str(period_s)] + pairs,
                           log=RUN_DIR.rstrip("/") + ".mirror.log", pid_file=MIRROR_PID)
    print(f"Drive mirror started (pid {pid}) every {period_s / 60:.0f} min -> {DRIVE_DIR}")


def restore_from_drive():
    """After a recycled machine: copy the mirrored run (and library, proxy outputs) back."""
    for src, dst in ((DRIVE_DIR, RUN_DIR),
                     (DRIVE_DIR + ".data_cache/library", f"{REPO_DIR}/data_cache/library"),
                     (DRIVE_DIR + ".data_cache/proxy", f"{REPO_DIR}/data_cache/proxy")):
        if os.path.isdir(src) and not os.path.exists(os.path.join(dst, "manifest.json" if dst == RUN_DIR else "")):
            print(f"restoring {src} -> {dst}")
            shutil.copytree(src, dst, dirs_exist_ok=True,
                            ignore=shutil.ignore_patterns("*.part", ".tmp-*"))


def _read_json(path):
    try:
        with open(path) as f:
            return json.load(f)
    except (OSError, ValueError):
        return None


def _secs(x):
    return "-" if not isinstance(x, (int, float)) else f"{x / 3600:.2f} h" if x >= 3600 else f"{x / 60:.1f} min" if x >= 60 else f"{x:.0f} s"


def show_progress(run_dir=RUN_DIR, tail=12):
    """Stage table, benchmark and canary, the last training steps, background tasks, the log."""
    print(f"run {'RUNNING' if pid_alive() else 'NOT running'} ({run_dir})")
    man = _read_json(os.path.join(run_dir, "manifest.json")) or {}
    st = _read_json(os.path.join(run_dir, "stages.json")) or {}
    b = man.get("budget") or {}
    if b:
        left = b["deadline"] - time.time()
        print(f"budget {_secs(b['budget_s'])}, elapsed {_secs(time.time() - b['t0'])}, left {_secs(max(0.0, left))}")
    print("\n stage        status     planned     used  note")
    for name in ("preflight", "background", "benchmark", "train", "sample", "critic", "prefopt",
                 "select", "judge", "export"):
        r = (st.get("stages") or {}).get(name) or {}
        print(f" {name:12s} {r.get('status', '-'):9s} {_secs(r.get('planned_s')):>9s} {_secs(r.get('used_s')):>8s}  {(r.get('reason') or '')[:70]}")
    bench = man.get("benchmark") or {}
    if bench:
        print(f"\nbenchmark: plan {bench.get('plan_key')}, expected {bench.get('expected_tps', 0):,.0f} tok/s, horizon {bench.get('horizon_tokens', 0):.3e} tokens")
        for m in bench.get("measurements") or []:
            p = m.get("plan") or {}
            key = f"{p.get('attn_impl')}-remat{int(bool(p.get('remat')))}-ms{p.get('micro_seqs')}-{p.get('table_grad')}"
            peak = (m.get("peak_bytes") or 0) / 2 ** 30
            print(f"  {key:28s} T={m.get('seq_len')}: {'ok ' if m.get('ok') else 'NO '} compile {m.get('compile_s') or 0:6.1f} s, step {m.get('step_s') or 0:6.3f} s, {m.get('tokens_per_s') or 0:10,.0f} tok/s, peak {peak:5.2f} GiB {('' if m.get('ok') else m.get('reason', ''))[:60]}")
        can = bench.get("canary") or {}
        lad = bench.get("ladder") or {}
        print("  canary: " + ", ".join(f"{k} {(can.get(k) or {}).get('ok')}" for k in ("parity", "determinism", "safe", "full")))
        print(f"  fallback ladder rung {lad.get('rung')}" + (f": {', '.join(lad.get('groups') or [])} off" if lad.get("groups") else " (full configuration)"))
        h = bench.get("host") or {}
        print(f"  host loader {h.get('rows_per_s', 0):,.0f} rows/s with {h.get('workers')} workers, device needs {h.get('needed_rows_per_s', 0):,.0f}; workers for training {h.get('loader_workers')}")
    led = _read_json(os.path.join(run_dir, "train", "ledger.json")) or {}
    if led:
        s = led.get("seconds") or {}
        print(f"\ntraining ledger: {led.get('tokens', 0):,} tokens, {led.get('steps', 0):,} steps, train {_secs(s.get('train'))}, lost {_secs(s.get('lost'))}, restarts {led.get('restarts')}, rollbacks {led.get('rollbacks')}")
    path = os.path.join(run_dir, "train", "train_log.jsonl")
    if os.path.exists(path):
        steps, events = [], []
        with open(path) as f:
            for line in f:
                try:
                    r = json.loads(line)
                except ValueError:
                    continue
                (steps if r.get("event") == "step" else events).append(r)
        for r in steps[-5:]:
            print(f"  step {r['step']:7d}  p {r['p']:.4f}  T {r.get('seq_len')}  loss {r['loss']:.4f}  ema {r.get('loss_ema', float('nan')):.4f}  {r.get('tok_per_s', 0):10,.0f} tok/s  lr x{r.get('lr', 0):.3g}  {r.get('action', '')}")
        for r in events[-4:]:
            print(f"  event {r.get('event')}: " + ", ".join(f"{k}={r[k]}" for k in ("step", "reason", "path", "p") if k in r))
    bg = (_read_json(os.path.join(run_dir, "background.json")) or {}).get("tasks") or {}
    if bg:
        print("\nbackground: " + ", ".join(f"{k} {v.get('state')}" + (f" ({v.get('error')[:40]})" if v.get("error") else "") for k, v in bg.items()))
    hl = _read_json(os.path.join(run_dir, "health", "latest.json"))
    if hl:
        print(f"side-car (step {(hl.get('checkpoint') or {}).get('step')}): probe loss {hl.get('probe_loss')}")
    if os.path.exists(LOG):
        with open(LOG, errors="replace") as f:
            lines = f.readlines()[-tail:]
        print("\n--- log tail ---\n" + "".join(lines))

In [ ]:
# 2. runtime check (in a separate process: this kernel must not take the TPU) -----------------------
import json, os, shutil, subprocess, sys

probe = ("import json, jax, jaxlib; d = jax.devices(); print(json.dumps({'jax': jax.__version__, "
         "'jaxlib': jaxlib.__version__, 'backend': jax.default_backend(), 'kinds': [x.device_kind for x in d]}))")
out = subprocess.run([sys.executable, "-c", probe], capture_output=True, text=True, timeout=900)
if out.returncode != 0:
    raise RuntimeError("JAX does not start on this runtime:\n" + out.stderr[-3000:])
JAX_INFO = json.loads(out.stdout.strip().splitlines()[-1])
print(JAX_INFO)
if JAX_INFO["backend"] != "tpu" and not ALLOW_CPU:
    raise RuntimeError(f"no TPU (backend {JAX_INFO['backend']}): choose Runtime > Change runtime type > TPU v6e-1")
if JAX_INFO["backend"] == "tpu" and not any("v6" in k for k in JAX_INFO["kinds"]):
    raise RuntimeError(f"this is not a TPU v6e ({JAX_INFO['kinds']}): the memory plan and the time budget assume one v6e chip")
if len(JAX_INFO["kinds"]) != 1:
    print(f"WARNING: {len(JAX_INFO['kinds'])} devices; the run uses device 0 only")
if JAX_INFO["jax"] != "0.11.1":
    print(f"WARNING: the runtime has jax {JAX_INFO['jax']}; the code was compile-checked with 0.11.1. "
          "It is not upgraded or downgraded here; the run's preflight records the versions.")
mem_gb = 0.0
for line in open("/proc/meminfo"):
    if line.startswith("MemTotal:"):
        mem_gb = int(line.split()[1]) / 1e6
du = shutil.disk_usage("/content")
print(f"{os.cpu_count()} host cores, {mem_gb:.0f} GB RAM, {du.free / 1e9:.0f} GB free disk under /content")
if du.free / 1e9 < 120:
    print("WARNING: the default plan needs about 111 GB of disk (downloads plus token shards); the "
          "preflight stops the run if the estimate does not fit (RUNBOOK.md: lower the download caps)")

In [ ]:
# 3. the code: clone, or use an existing copy (Drive) ---------------------------------------------
if USE_DRIVE or REPO_DIR.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
if os.path.isdir(os.path.join(REPO_DIR, "bes")):
    print(f"using the code in {REPO_DIR}")
else:
    if not REPO_URL:
        raise RuntimeError("set REPO_URL (and BRANCH) in cell 1, or put a copy of the repository at REPO_DIR")
    sh(["git", "clone", "--depth", "1"] + (["--branch", BRANCH] if BRANCH else []) + [REPO_URL, REPO_DIR])
os.chdir(REPO_DIR)
print(subprocess.run(["git", "log", "-1", "--format=%H %ad %s"], capture_output=True, text=True).stdout)

In [ ]:
# 4. dependencies (requirements-colab.txt) without touching the preinstalled jax ------------------
# llama-cpp-python is built from source: this cell can take 5 to 15 minutes.
pin_code = ("import importlib.metadata as m\n"
            "for n in ('jax', 'jaxlib', 'libtpu', 'libtpu-nightly'):\n"
            "    try:\n        print(n + '==' + m.version(n))\n"
            "    except m.PackageNotFoundError:\n        pass\n")
held = subprocess.run([sys.executable, "-c", pin_code], capture_output=True, text=True).stdout.split()
with open("/content/constraints-jax.txt", "w") as f:
    f.write("\n".join(held) + "\n")
print("held at their installed versions:", held)
env = dict(os.environ, CMAKE_BUILD_PARALLEL_LEVEL=str(os.cpu_count() or 4))
sh([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt",
    "-c", "/content/constraints-jax.txt"], env=env)
after = subprocess.run([sys.executable, "-c", "import jax, jaxlib; print(jax.__version__, jaxlib.__version__)"],
                       capture_output=True, text=True).stdout.split()
if after != [JAX_INFO["jax"], JAX_INFO["jaxlib"]]:
    raise RuntimeError(f"jax changed during the install ({JAX_INFO['jax']} -> {after}): restart the runtime and investigate")
check = subprocess.run([sys.executable, "-c", "import llama_cpp, optax, tiktoken, rustbpe, pyarrow, huggingface_hub, regex; print('imports ok')"],
                       capture_output=True, text=True)
print(check.stdout or check.stderr[-2000:])

In [ ]:
# 5. Hugging Face token (kept in this kernel's memory only, never written to the notebook) ---------
from getpass import getpass

token = ""
try:
    from google.colab import userdata          # a Colab secret named HF_TOKEN, if you granted access
    token = userdata.get("HF_TOKEN") or ""
except Exception:
    pass
if not token:
    token = getpass("Hugging Face token (Enter to skip): ").strip()
if token:
    os.environ["HF_TOKEN"] = token
del token
print("HF_TOKEN", "set" if os.environ.get("HF_TOKEN") else "not set: public files only, lower rate limits")

In [ ]:
# 6. start the 24-hour run in the background --------------------------------------------------------
os.makedirs(os.path.dirname(RUN_DIR), exist_ok=True)
if pid_alive():
    raise RuntimeError("a run started from this notebook is still running (cell 7 shows it)")
if os.path.exists(os.path.join(RUN_DIR, "manifest.json")):
    raise RuntimeError(f"{RUN_DIR} already holds a run: continue it with cell 8, or choose another RUN_DIR")
argv = [sys.executable, "scripts/run_24h.py", "--config", CONFIG, "--out", RUN_DIR]
if BUDGET_MINUTES:
    argv += ["--budget-minutes", str(BUDGET_MINUTES)]
if ALLOW_CPU:
    argv += ["--allow-cpu"]
pid = start_background(argv)
print(f"started pid {pid}: {' '.join(argv)}\nlog: {LOG}")
start_mirror()

In [ ]:
# 7. progress (run this cell whenever you like) -----------------------------------------------------
show_progress()

In [ ]:
# 8. continue after a disconnect -------------------------------------------------------------------
# On a recycled machine run cells 1 to 5 (and 1b) first (code, dependencies, token), then this cell: it
# copies the mirrored run back from Drive and starts scripts/run_24h.py --resume (same budget:
# the downtime counts against it).
if pid_alive():
    print("the run is still running: nothing to do (cell 7 shows its progress)")
else:
    if not os.path.exists(os.path.join(RUN_DIR, "manifest.json")) and USE_DRIVE:
        restore_from_drive()
    if not os.path.exists(os.path.join(RUN_DIR, "manifest.json")):
        raise RuntimeError(f"no run to continue in {RUN_DIR} (and none mirrored in {DRIVE_DIR})")
    argv = [sys.executable, "scripts/run_24h.py", "--out", RUN_DIR, "--resume"]
    if ALLOW_CPU:
        argv += ["--allow-cpu"]
    pid = start_background(argv)
    print(f"resumed: pid {pid}; log {LOG}")
    start_mirror()

In [ ]:
# 9. (only if you must) stop the run cleanly: the trainer saves a checkpoint; cell 8 continues it ---
if pid_alive():
    os.kill(int(open(PID_FILE).read()), signal.SIGTERM)
    print("SIGTERM sent; the run stops after its current step and writes a final checkpoint")
else:
    print("the run is not running")

## After the run

`RUN_DIR/final/` holds the export (`MODEL_CARD.md` first: the held-out check of the 16-bit
weights, the accounting, the read-set audit, the stage table, the evaluation results that exist).
The chat and the arena below use it. If the run is still training they run on the host CPU (the
run owns the TPU); otherwise this kernel takes the TPU, so do not start another run from this
runtime afterwards without restarting it.

In [ ]:
# 10. chat with the exported model (type /quit to leave) ------------------------------------------
import importlib.util

EXPORT = os.path.join(RUN_DIR, "final")
if pid_alive():
    os.environ["JAX_PLATFORMS"] = "cpu"
    print("the run still owns the TPU: the chat runs on the CPU")
spec = importlib.util.spec_from_file_location("bes_chat", os.path.join(REPO_DIR, "scripts", "chat.py"))
bes_chat = importlib.util.module_from_spec(spec)
spec.loader.exec_module(bes_chat)
bes_chat.main(["--export", EXPORT, "--show-diagnostics"])

In [ ]:
# 11. blind arena: our model against the opponents (q quits; votes are saved after each one) --------
import importlib.util

REVEAL = False                     # True: show which model wrote which reply after each vote
PROMPTS = None                     # None: type prompts; or "proxy:test", or a .txt / .jsonl file
args = ["--export", os.path.join(RUN_DIR, "final"), "--name", "final"]
if REVEAL:
    args.append("--reveal")
if PROMPTS:
    args += ["--prompts", PROMPTS]
if pid_alive():
    os.environ["JAX_PLATFORMS"] = "cpu"
spec = importlib.util.spec_from_file_location("bes_arena", os.path.join(REPO_DIR, "scripts", "arena.py"))
bes_arena = importlib.util.module_from_spec(spec)
spec.loader.exec_module(bes_arena)
bes_arena.main(args)

In [ ]:
# 12. copy the export (and the arena votes) to Drive -----------------------------------------------
if USE_DRIVE:
    for src, dst in ((os.path.join(RUN_DIR, "final"), DRIVE_DIR + ".export"),
                     (os.path.join(REPO_DIR, "data_cache", "arena"), DRIVE_DIR + ".arena")):
        if os.path.isdir(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
            print(f"copied {src} -> {dst}")
print(subprocess.run([sys.executable, "scripts/export.py", "--verify", os.path.join(RUN_DIR, "final")],
                     capture_output=True, text=True).stdout)